In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/telco_churn.csv")
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(
    include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

C:\Users\denar\AppData\Local\Temp\ipykernel_29988\815205654.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Pré-processador com normalização (para LogReg e MLP)
preprocessor_scaled = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Pré-processador sem normalização (para Random Forest)
preprocessor_tree = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='passthrough')

pipelines = {
    'Regressao Logistica': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('classifier', RandomForestClassifier(
            n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))
    ]),
    'MLP': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', MLPClassifier(hidden_layer_sizes=(50, 25),
         max_iter=500, early_stopping=True, random_state=42))
    ])
}

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# 5 folds, mantendo a proporção de churn em cada um
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=cv, scoring='f1', n_jobs=-1)
    results[name] = scores
    print(f"{name}: F1 médio = {scores.mean():.4f} (+/- {scores.std():.4f})")
    print(f"  Scores individuais: {np.round(scores, 4)}\n")

Regressao Logistica: F1 médio = 0.5923 (+/- 0.0299)
  Scores individuais: [0.5861 0.5495 0.5863 0.6425 0.5974]

Random Forest: F1 médio = 0.5828 (+/- 0.0203)
  Scores individuais: [0.5687 0.563  0.6211 0.5805 0.5805]

MLP: F1 médio = 0.5940 (+/- 0.0227)
  Scores individuais: [0.5859 0.5596 0.6092 0.6268 0.5887]



In [4]:
from sklearn.metrics import f1_score, roc_auc_score

final_results = []

for name, pipeline in pipelines.items():
    # Treina no conjunto de treino completo
    pipeline.fit(X_train, y_train)

    # Avalia no conjunto de teste (nunca visto até agora)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    final_results.append({
        'Modelo': name,
        'F1-score (teste)': round(f1, 4),
        'AUC-ROC (teste)': round(auc, 4)
    })

results_df = pd.DataFrame(final_results).sort_values(
    'F1-score (teste)', ascending=False)
results_df

,Modelo,F1-score (teste),AUC-ROC (teste)
0,Regressao Logistica,0.6040,0.8421
1,Random Forest,0.5791,0.8364
2,MLP,0.5701,0.8430


In [5]:
import joblib

# Retreina o pipeline campeão com todo o conjunto de treino (já foi feito no loop anterior,
# mas deixamos explícito aqui para clareza do notebook)
champion_pipeline = pipelines['Regressao Logistica']
champion_pipeline.fit(X_train, y_train)

# Salva o pipeline completo (pré-processamento + modelo) em um único arquivo
joblib.dump(champion_pipeline, '../models/churn_model.joblib')

print("Modelo campeão salvo em models/churn_model.joblib")

Modelo campeão salvo em models/churn_model.joblib


## Conclusão: Modelo Campeão

Após treinar três modelos (Regressão Logística, Random Forest e MLPClassifier) e
validá-los com validação cruzada estratificada (5 folds) e avaliação final em
conjunto de teste isolado, o modelo escolhido como campeão foi a **Regressão
Logística**.

**Justificativa:**
- Melhor F1-score na avaliação final no conjunto de teste (0.6040 vs. 0.5791
  do Random Forest e 0.5701 do MLP).
- Performance competitiva também na validação cruzada (F1 médio de 0.5923),
  dentro da margem de variação dos demais modelos — não houve vantagem
  estatisticamente clara de modelos mais complexos sobre o baseline linear.
- Maior interpretabilidade e menor custo computacional, características
  relevantes para explicabilidade junto à área de negócio e para manutenção
  do modelo em produção.

O modelo foi salvo em `models/churn_model.joblib`, incluindo todo o pipeline
de pré-processamento (normalização e codificação categórica), garantindo
consistência entre o treinamento e a futura API de inferência.